In [1]:
import os
import pdal
import richdem as rd
import numpy as np
import json
import laspy
from pyproj import CRS
from config import *
import rasterio
from pathlib import Path
from osgeo import gdal
from sklearn.neighbors import KNeighborsRegressor
from whitebox.whitebox_tools import WhiteboxTools

In [2]:
data_dir = '/home/ajai-krishna/work/GEO_AI/data/'
out_dir = '/home/ajai-krishna/work/GEO_AI/outputs/subsets'
output_training_tif = '/home/ajai-krishna/work/GEO_AI/data/input/subsets'

In [3]:

crs = CRS.from_proj4("+proj=utm +zone=43 +datum=WGS84 +units=m +no_defs")

In [4]:
gujarath_data = '/home/ajai-krishna/work/GEO_AI/data/input/laz/DEVDI_POINT CLOUD (511671).las'
# gujarath_las = laspy.open(gujarath_data_path)

In [5]:
# with laspy.open(gujarath_data_path) as gujarath_las:
#     gujarath_las_header = gujarath_las.header
#     print(list(gujarath_las_header.point_format.dimension_names))
#     las_x = gujarath_las.X
#     las_y = gujarath_las.Y
#     las_z = gujarath_las.Z
#     las_rgb = gujarath_las.red, gujarath_las.green, gujarath_las.blue
#     for 

In [6]:
# gujarath_las_header = gujarath_las.header
# print(list(gujarath_las_header.point_format.dimension_names))

In [7]:
# las_x = gujarath_las.X
# las_y = gujarath_las.Y
# las_z = gujarath_las.Z
# las_rgb = gujarath_las.red, gujarath_las.green, gujarath_las.blue


In [8]:
# def extract_dtm_from_las(las_file, output_path, outfile_name="output", chunk_size=1_000_000):

#     output_path = Path(output_path)
#     output_path.mkdir(parents=True, exist_ok=True)

#     r_files = []
#     g_files = []
#     b_files = []

#     with laspy.open(las_file) as reader:

#         header = reader.header
#         chunk_id = 0

#         for points in reader.chunk_iterator(chunk_size):

#             chunk_id += 1

#             temp_las = output_path / f"chunk_{chunk_id}.las"

#             # write temporary chunk
#             with laspy.open(temp_las, mode="w", header=header) as writer:
#                 writer.write_points(points)

#             output_r = output_path / f"{outfile_name}_chunk{chunk_id}_R.tif"
#             output_g = output_path / f"{outfile_name}_chunk{chunk_id}_G.tif"
#             output_b = output_path / f"{outfile_name}_chunk{chunk_id}_B.tif"

#             pdal_pipeline = {
#                 "pipeline": [
#                     {
#                         "type": "readers.las",
#                         "filename": str(temp_las)
#                     },
#                     {
#                         "type": "filters.smrf"
#                     },
#                     {
#                         "type": "filters.range",
#                         "limits": "Classification[2:2]"
#                     },
#                     {
#                         "type": "writers.gdal",
#                         "filename": str(output_r),
#                         "dimension": "Red",
#                         "resolution": 0.09,
#                         "output_type": "mean",
#                         "data_type": "float32"
#                     },
#                     {
#                         "type": "writers.gdal",
#                         "filename": str(output_g),
#                         "dimension": "Green",
#                         "resolution": 0.09,
#                         "output_type": "mean",
#                         "data_type": "float32"
#                     },
#                     {
#                         "type": "writers.gdal",
#                         "filename": str(output_b),
#                         "dimension": "Blue",
#                         "resolution": 0.09,
#                         "output_type": "mean",
#                         "data_type": "float32"
#                     }
#                 ]
#             }

#             pipeline = pdal.Pipeline(json.dumps(pdal_pipeline))
#             pipeline.execute()

#             r_files.append(str(output_r))
#             g_files.append(str(output_g))
#             b_files.append(str(output_b))

#             print(f"Chunk {chunk_id} processed")

#             temp_las.unlink()

#     # -----------------------------
#     # STACK RGB INTO FINAL TIFF
#     # -----------------------------

#     final_rgb = output_path / f"{outfile_name}_RGB.tif"

#     vrt = output_path / "rgb_stack.vrt"

#     gdal.BuildVRT(str(vrt), r_files + g_files + b_files, separate=True)
#     gdal.Translate(str(final_rgb), str(vrt))

#     print("Final stacked RGB raster saved at:")
#     print(final_rgb)

In [9]:
# def knn_fill_raster(input_tif, output_tif, k=5):

#     with rasterio.open(input_tif) as src:
#         data = src.read(1)
#         profile = src.profile
#         nodata = src.nodata

#     if nodata is None:
#         nodata = np.nan

#     mask_valid = ~np.isnan(data) if np.isnan(nodata) else data != nodata
#     mask_missing = ~mask_valid

#     rows, cols = np.indices(data.shape)

#     X_train = np.column_stack((rows[mask_valid], cols[mask_valid]))
#     y_train = data[mask_valid]

#     X_pred = np.column_stack((rows[mask_missing], cols[mask_missing]))

#     if len(X_pred) == 0:
#         print("No holes found.")
#         return

#     knn = KNeighborsRegressor(n_neighbors=k, weights="distance")
#     knn.fit(X_train, y_train)

#     predicted = knn.predict(X_pred)

#     filled = data.copy()
#     filled[mask_missing] = predicted

#     with rasterio.open(output_tif, "w", **profile) as dst:
#         dst.write(filled, 1)

#     print("KNN hole filling completed:", output_tif)

In [10]:
# def extract_dtm_from_las(las_file, output_path, outfile_name="output", chunk_size=3_000_000):

#     output_path = Path(output_path)
#     output_path.mkdir(parents=True, exist_ok=True)

#     r_files = []
#     g_files = []
#     b_files = []
#     z_files = []

#     with laspy.open(las_file) as reader:

#         header = reader.header
#         chunk_id = 0

#         for points in reader.chunk_iterator(chunk_size):

#             chunk_id += 1

#             temp_las = output_path / f"chunk_{chunk_id}.las"

#             with laspy.open(temp_las, mode="w", header=header) as writer:
#                 writer.write_points(points)

#             output_r = output_path / f"{outfile_name}_chunk{chunk_id}_R.tif"
#             output_g = output_path / f"{outfile_name}_chunk{chunk_id}_G.tif"
#             output_b = output_path / f"{outfile_name}_chunk{chunk_id}_B.tif"
#             output_z = output_path / f"{outfile_name}_chunk{chunk_id}_Z.tif"

#             pdal_pipeline = {
#                 "pipeline": [
#                     {"type": "readers.las", "filename": str(temp_las)},
#                     { "type"           : "filters.smrf"},
#                     #   "class_threshold": 0.3},
#                     {"type": "filters.range", "limits": "Classification[2:2]"},
#                     {
#                         "type": "writers.gdal",
#                         "filename": str(output_r),
#                         "dimension": "Red",
#                         "resolution": 0.09,
#                         "output_type": "mean",
#                         "data_type": "float32"
#                     },
#                     {
#                         "type": "writers.gdal",
#                         "filename": str(output_g),
#                         "dimension": "Green",
#                         "resolution": 0.09,
#                         "output_type": "mean",
#                         "data_type": "float32"
#                     },
#                     {
#                         "type": "writers.gdal",
#                         "filename": str(output_b),
#                         "dimension": "Blue",
#                         "resolution": 0.09,
#                         "output_type": "mean",
#                         "data_type": "float32"
#                     },
#                     {
#                         "type": "writers.gdal",
#                         "filename": str(output_z),
#                         "dimension": "Z",
#                         "resolution": 0.09,
#                         "output_type": "mean",
#                         "data_type": "float32"}
#                 ]
#             }

#             pipeline = pdal.Pipeline(json.dumps(pdal_pipeline))
#             pipeline.execute()

#             r_files.append(str(output_r))
#             g_files.append(str(output_g))
#             b_files.append(str(output_b))
#             z_files.append(str(output_z))

#             print(f"Chunk {chunk_id} processed")

#             temp_las.unlink()

#     # -----------------------------
#     # STEP 1: Mosaic each band separately
#     # -----------------------------

#     mosaic_r = output_path / f"{outfile_name}_mosaic_R.tif"
#     mosaic_g = output_path / f"{outfile_name}_mosaic_G.tif"
#     mosaic_b = output_path / f"{outfile_name}_mosaic_B.tif"
#     mosaic_z = output_path / f"{outfile_name}_mosaic_Z.tif"

#     vrt_r = output_path / "mosaic_R.vrt"
#     vrt_g = output_path / "mosaic_G.vrt"
#     vrt_b = output_path / "mosaic_B.vrt"
#     vrt_z = output_path / "mosaic_Z.vrt"

#     # Mosaic all R chunks into one raster
#     gdal.BuildVRT(str(vrt_r), r_files)          # No separate=True → spatial mosaic
#     gdal.Translate(str(mosaic_r), str(vrt_r))

#     # Mosaic all G chunks into one raster
#     gdal.BuildVRT(str(vrt_g), g_files)
#     gdal.Translate(str(mosaic_g), str(vrt_g))

#     # Mosaic all B chunks into one raster
#     gdal.BuildVRT(str(vrt_b), b_files)
#     gdal.Translate(str(mosaic_b), str(vrt_b))

#     # Mosaic all Z chunks into one raster
#     gdal.BuildVRT(str(vrt_z), z_files)
#     gdal.Translate(str(mosaic_z), str(vrt_z))

#     print("Per-band mosaics created")

#     # -----------------------------
#     # STEP 2: Stack R, G, B mosaics into final RGB
#     # -----------------------------

#     final_rgb = output_path / f"{outfile_name}_RGB.tif"
#     vrt_rgb   = output_path / "rgb_stack.vrt"

#     vrt_options = gdal.BuildVRTOptions(separate=True)
#     gdal.BuildVRT(str(vrt_rgb), [str(mosaic_r), str(mosaic_g), str(mosaic_b), str(mosaic_z)], options=vrt_options)

#     # Compress the VRT itself
#     compressed_vrt = output_path / "rgb_stack_compressed.vrt"
#     gdal.Translate(
#         str(compressed_vrt),
#         str(vrt_rgb),
#         format="VRT",
#         creationOptions=["COMPRESS=LZW"]
#     )

#     # Translate compressed VRT → final RGB GeoTIFF
#     gdal.Translate(
#         str(final_rgb),
#         str(compressed_vrt),
#         creationOptions=["TILED=YES", "COMPRESS=LZW", "PHOTOMETRIC=RGB"]
#     )

#     print("Compressed VRT saved at:")
#     print(compressed_vrt)
#     print("Final stacked RGB raster saved at:")
#     print(final_rgb)

In [11]:
def extract_dsm_tif_from_las(las_file,output_path,outfile_name="dsm_training_output",chunk_size=3_000_000):
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    r_files = []
    g_files = []
    b_files = []
    z_files = []

    with laspy.open(las_file) as reader:

        header = reader.header
        chunk_id = 0

        for points in reader.chunk_iterator(chunk_size):

            chunk_id += 1

            temp_las = output_path / f"chunk_{chunk_id}.las"

            with laspy.open(temp_las, mode="w", header=header) as writer:
                writer.write_points(points)

            output_r = output_path / f"{outfile_name}_chunk{chunk_id}_R.tif"
            output_g = output_path / f"{outfile_name}_chunk{chunk_id}_G.tif"
            output_b = output_path / f"{outfile_name}_chunk{chunk_id}_B.tif"
            output_z = output_path / f"{outfile_name}_chunk{chunk_id}_Z.tif"
            pdal_pipeline = {
                "pipeline": [
                    {"type": "readers.las", "filename": str(temp_las)},
                    {
                        "type": "writers.gdal",
                        "filename": str(output_r),
                        "dimension": "Red",
                        "resolution": 0.09,
                        "output_type": "max",
                        "data_type": "float32"
                    },
                    {
                        "type": "writers.gdal",
                        "filename": str(output_g),
                        "dimension": "Green",
                        "resolution": 0.09,
                        "output_type": "max",
                        "data_type": "float32"
                    },
                    {
                        "type": "writers.gdal",
                        "filename": str(output_b),
                        "dimension": "Blue",
                        "resolution": 0.09,
                        "output_type": "max",
                        "data_type": "float32"
                    },
                    {
                        "type": "writers.gdal",
                        "filename": str(output_z),
                        "dimension": "Z",
                        "resolution": 0.09,
                        "output_type": "max",
                        "data_type": "float32"}
                ]
            }
            pipeline = pdal.Pipeline(json.dumps(pdal_pipeline))
            pipeline.execute()
            r_files.append(str(output_r))
            g_files.append(str(output_g))
            b_files.append(str(output_b))
            z_files.append(str(output_z))

            print(f"Chunk {chunk_id} processed")

            temp_las.unlink()

    # -----------------------------
    # STEP 1: Mosaic each band separately
    # -----------------------------

    mosaic_r = output_path / f"{outfile_name}_mosaic_R.tif"
    mosaic_g = output_path / f"{outfile_name}_mosaic_G.tif"
    mosaic_b = output_path / f"{outfile_name}_mosaic_B.tif"
    mosaic_z = output_path / f"{outfile_name}_mosaic_Z.tif"

    vrt_r = output_path / "mosaic_R.vrt"
    vrt_g = output_path / "mosaic_G.vrt"
    vrt_b = output_path / "mosaic_B.vrt"
    vrt_z = output_path / "mosaic_Z.vrt"

    # Mosaic all R chunks into one raster
    gdal.BuildVRT(str(vrt_r), r_files)          # No separate=True → spatial mosaic
    gdal.Translate(str(mosaic_r), str(vrt_r))

    # Mosaic all G chunks into one raster
    gdal.BuildVRT(str(vrt_g), g_files)
    gdal.Translate(str(mosaic_g), str(vrt_g))

    # Mosaic all B chunks into one raster
    gdal.BuildVRT(str(vrt_b), b_files)
    gdal.Translate(str(mosaic_b), str(vrt_b))

    # Mosaic all Z chunks into one raster
    gdal.BuildVRT(str(vrt_z), z_files)
    gdal.Translate(str(mosaic_z), str(vrt_z))

    print("Per-band mosaics created")

    # -----------------------------
    # STEP 2: Stack R, G, B mosaics into final RGB
    # -----------------------------

    final_rgb = output_path / f"{outfile_name}_RGBZ.tif"
    vrt_rgb   = output_path / "rgb_stack.vrt"

    vrt_options = gdal.BuildVRTOptions(separate=True)
    gdal.BuildVRT(str(vrt_rgb), [str(mosaic_r), str(mosaic_g), str(mosaic_b), str(mosaic_z)], options=vrt_options)

    # Compress the VRT itself
    compressed_vrt = output_path / "rgb_stack_compressed.vrt"
    gdal.Translate(
        str(compressed_vrt),
        str(vrt_rgb),
        format="VRT",
        creationOptions=["COMPRESS=LZW"]
    )

    # Translate compressed VRT → final RGB GeoTIFF
    gdal.Translate(
        str(final_rgb),
        str(compressed_vrt),
        creationOptions=["TILED=YES", "COMPRESS=LZW", "PHOTOMETRIC=RGB"]
    )

    print("Compressed VRT saved at:")
    print(compressed_vrt)
    print("Final stacked RGB raster saved at:")
    print(final_rgb)
            


In [12]:
def extract_dtm_from_las(las_file, output_path, outfile_name="output", chunk_size=3_000_000):

    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    r_files = []
    g_files = []
    b_files = []
    z_files = []

    with laspy.open(las_file) as reader:

        header = reader.header
        chunk_id = 0

        for points in reader.chunk_iterator(chunk_size):

            chunk_id += 1

            temp_las = output_path / f"chunk_{chunk_id}.las"

            with laspy.open(temp_las, mode="w", header=header) as writer:
                writer.write_points(points)

            output_r = output_path / f"{outfile_name}_chunk{chunk_id}_R.tif"
            output_g = output_path / f"{outfile_name}_chunk{chunk_id}_G.tif"
            output_b = output_path / f"{outfile_name}_chunk{chunk_id}_B.tif"
            output_z = output_path / f"{outfile_name}_chunk{chunk_id}_Z.tif"

            pdal_pipeline = {
                "pipeline": [
                    {"type": "readers.las", "filename": str(temp_las)},
                    { "type"           : "filters.smrf"},
                    #   "class_threshold": 0.3},
                    {"type": "filters.range", "limits": "Classification[2:2]"},
                    {
                        "type": "writers.gdal",
                        "filename": str(output_r),
                        "dimension": "Red",
                        "resolution": 0.09,
                        "output_type": "mean",
                        "data_type": "float32"
                    },
                    {
                        "type": "writers.gdal",
                        "filename": str(output_g),
                        "dimension": "Green",
                        "resolution": 0.09,
                        "output_type": "mean",
                        "data_type": "float32"
                    },
                    {
                        "type": "writers.gdal",
                        "filename": str(output_b),
                        "dimension": "Blue",
                        "resolution": 0.09,
                        "output_type": "mean",
                        "data_type": "float32"
                    },
                    {
                        "type": "writers.gdal",
                        "filename": str(output_z),
                        "dimension": "Z",
                        "resolution": 0.09,
                        "output_type": "mean",
                        "data_type": "float32"}
                ]
            }

            pipeline = pdal.Pipeline(json.dumps(pdal_pipeline))
            pipeline.execute()

            r_files.append(str(output_r))
            g_files.append(str(output_g))
            b_files.append(str(output_b))
            z_files.append(str(output_z))

            print(f"Chunk {chunk_id} processed")

            temp_las.unlink()

    # -----------------------------
    # STEP 1: Mosaic each band separately
    # -----------------------------

    mosaic_r = output_path / f"{outfile_name}_mosaic_R.tif"
    mosaic_g = output_path / f"{outfile_name}_mosaic_G.tif"
    mosaic_b = output_path / f"{outfile_name}_mosaic_B.tif"
    mosaic_z = output_path / f"{outfile_name}_mosaic_Z.tif"

    vrt_r = output_path / "mosaic_R.vrt"
    vrt_g = output_path / "mosaic_G.vrt"
    vrt_b = output_path / "mosaic_B.vrt"
    vrt_z = output_path / "mosaic_Z.vrt"

    # Mosaic all R chunks into one raster
    gdal.BuildVRT(str(vrt_r), r_files)          # No separate=True → spatial mosaic
    gdal.Translate(str(mosaic_r), str(vrt_r))

    # Mosaic all G chunks into one raster
    gdal.BuildVRT(str(vrt_g), g_files)
    gdal.Translate(str(mosaic_g), str(vrt_g))

    # Mosaic all B chunks into one raster
    gdal.BuildVRT(str(vrt_b), b_files)
    gdal.Translate(str(mosaic_b), str(vrt_b))

    # Mosaic all Z chunks into one raster
    gdal.BuildVRT(str(vrt_z), z_files)
    gdal.Translate(str(mosaic_z), str(vrt_z))

    filled_z = output_path / f"{outfile_name}_Z_filled.tif"

    # knn_fill_raster(mosaic_z, filled_z, k=5)

    print("DTM holes filled using KNN")

    # -----------------------------
    # STEP 2: Stack R, G, B mosaics into final RGB
    # -----------------------------

    final_rgb = output_path / f"{outfile_name}_RGB.tif"
    vrt_rgb   = output_path / "rgb_stack.vrt"

    vrt_options = gdal.BuildVRTOptions(separate=True)
    gdal.BuildVRT(
    str(vrt_rgb),
    [str(mosaic_r), str(mosaic_g), str(mosaic_b), str(filled_z)],
    options=vrt_options
)

    # Compress the VRT itself
    compressed_vrt = output_path / "rgb_stack_compressed.vrt"
    gdal.Translate(
        str(compressed_vrt),
        str(vrt_rgb),
        format="VRT",
        creationOptions=["COMPRESS=LZW"]
    )

    # Translate compressed VRT → final RGB GeoTIFF
    gdal.Translate(
        str(final_rgb),
        str(compressed_vrt),
        creationOptions=["TILED=YES", "COMPRESS=LZW", "PHOTOMETRIC=RGB"]
    )

    print("Compressed VRT saved at:")
    print(compressed_vrt)
    print("Final stacked RGB raster saved at:")
    print(final_rgb)

In [13]:
extract_dtm_from_las(gujarath_data, out_dir, "gujarath_dtm")

Chunk 1 processed
Chunk 2 processed
Chunk 3 processed
Chunk 4 processed
Chunk 5 processed
Chunk 6 processed
Chunk 7 processed
Chunk 8 processed
Chunk 9 processed
Chunk 10 processed
Chunk 11 processed
Chunk 12 processed
Chunk 13 processed
Chunk 14 processed
Chunk 15 processed
Chunk 16 processed
Chunk 17 processed
Chunk 18 processed
Chunk 19 processed
Chunk 20 processed
Chunk 21 processed
Chunk 22 processed
DTM holes filled using KNN


Warning 1: Can't open /home/ajai-krishna/work/GEO_AI/outputs/subsets/gujarath_dtm_Z_filled.tif. Skipping it
Warning 6: driver VRT does not support creation option COMPRESS


Compressed VRT saved at:
/home/ajai-krishna/work/GEO_AI/outputs/subsets/rgb_stack_compressed.vrt
Final stacked RGB raster saved at:
/home/ajai-krishna/work/GEO_AI/outputs/subsets/gujarath_dtm_RGB.tif


In [14]:
extract_dsm_tif_from_las(gujarath_data,output_training_tif,"gujarath_dsm")

Chunk 1 processed
Chunk 2 processed
Chunk 3 processed
Chunk 4 processed
Chunk 5 processed
Chunk 6 processed
Chunk 7 processed
Chunk 8 processed
Chunk 9 processed
Chunk 10 processed
Chunk 11 processed
Chunk 12 processed
Chunk 13 processed
Chunk 14 processed
Chunk 15 processed
Chunk 16 processed
Chunk 17 processed
Chunk 18 processed
Chunk 19 processed
Chunk 20 processed
Chunk 21 processed
Chunk 22 processed
Per-band mosaics created


Warning 6: driver VRT does not support creation option COMPRESS


Compressed VRT saved at:
/home/ajai-krishna/work/GEO_AI/data/input/subsets/rgb_stack_compressed.vrt
Final stacked RGB raster saved at:
/home/ajai-krishna/work/GEO_AI/data/input/subsets/gujarath_dsm_RGBZ.tif


In [15]:
input_tiff_path = '/home/ajai-krishna/work/GEO_AI/outputs/subsets/gujarath_dtm_chunk10_Z.tif'
output_translate_dir = '/home/ajai-krishna/work/GEO_AI/outputs/wbt_'
output_rd = os.path.join(output_translate_dir, "gujarath_dtm_rd_filled.tif")
output_rd

'/home/ajai-krishna/work/GEO_AI/outputs/wbt_/gujarath_dtm_rd_filled.tif'

In [16]:
output_translate_dir

'/home/ajai-krishna/work/GEO_AI/outputs/wbt_'

In [17]:
# from osgeo import gdal

# def fill_smrf_nodata(smrf_dtm_tif, output_tif, max_search_dist=100, smoothing=2):

#     gdal.Translate(output_tif, smrf_dtm_tif)

#     ds   = gdal.Open(output_tif, gdal.GA_Update)
#     band = ds.GetRasterBand(1)
#     mask = band.GetMaskBand()

#     gdal.FillNodata(
#         targetBand          = band,
#         maskBand            = mask,
#         maxSearchDist       = max_search_dist,
#         smoothingIterations = smoothing
#     )

#     band.FlushCache()
#     ds = None
#     print(f"SMRF nodata filled → {output_tif}")


In [18]:
# fill_smrf_nodata(input_tiff_path, os.path.join(output_translate_dir, "gujarath_dtm_RGBZ_filled.tif"))

In [19]:
dtm = rd.LoadGDAL(input_tiff_path)
filled = rd.FillDepressions(dtm)
rd.SaveGDAL(output_rd, filled)



A Priority-Flood (Zhou2016 version)
C Zhou, G., Sun, Z., Fu, S., 2016. An efficient variant of the Priority-Flood algorithm for filling depressions in raster digital elevation models. Computers & Geosciences 90, Part A, 87 – 96. doi:http://dx.doi.org/10.1016/j.cageo.2016.02.021

t Zhou2016 wall-time = 0.342337 s
